In [ ]:
# =============================================================================
# mc_block_3 — Monte Carlo: Hysteresis, Reform, and Stimulus Trap
#
# Purpose:
#   Populate and analyze the BISTABLE regime where W*(psi_L) < 0 < W*(1).
#
# Deliverables (computed on the bistable subset, per sweep cell):
#   D1: Δψ_crit(t) along the no-intervention institutional drift path
#   D2: Stimulus attenuation: exact Δg*(ψ; ΔΓ) = g*(ψ,Γ+ΔΓ) - g*(ψ,Γ)
#   D3: Escape probability under repeated reforms (Δψ, t_start, cadence, cap)
#
# Numerical sensitivity axes:
#   psi_grid_n, barpsi_bisect_iters, include_structural_drag, psi0_buffer,
#   N_used, dGamma, reform_freq, max_reforms
#
# Theory compliance:
#   - include_structural_drag=True  uses Γ_eff(ψ) = Γ - D(ψ) in g*.
#   - include_structural_drag=False uses Γ_eff(ψ) = Γ (Section 4.2 closure only).
#   Both specifications are reported.
# =============================================================================

import os
import json
import math
import platform
from dataclasses import dataclass
from datetime import datetime

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from scipy.interpolate import interp1d


# ----------------------------
# CONFIG
# ----------------------------
MC_NAME = "mc_block_3"
RUN_TS = datetime.now().strftime("%y%m%d_%H%M")

BASE_OUT_DIR = os.getcwd()
OUT_DIR = os.path.join(BASE_OUT_DIR, f"{MC_NAME}_{RUN_TS}")
FIGS_DIR = os.path.join(OUT_DIR, "figures")
os.makedirs(OUT_DIR, exist_ok=True)
os.makedirs(FIGS_DIR, exist_ok=True)

SEED = 20260312
rng = np.random.default_rng(SEED)

# High-basin start cap: must be feasible distance above barψ
PSI_HI = 0.999

# ψ0 cap parameters
EPS_PSI_L = 0.02
EPS_PSI_HI = 1e-4

# Bistability margin: prevent knife-edge regime classification
BISTABLE_MARGIN = 1e-3  # Require W*(psi_L) <= -eps and W*(1) >= eps

CFG = {
    # Pool build
    "N_BISTABLE_POOL_TARGET": 600,
    "MAX_TRIES": 400000,

    # Numerical sensitivity axes
    "psi_grid_n_list": [101, 201, 401],
    "barpsi_bisect_iters_list": [30, 60, 90],
    "include_structural_drag_list": [False, True],
    "psi0_buffer_list": [0.01, 0.03, 0.05],
    "N_used_list": [100, 300],

    # D3 evaluation budget (separate from D1/D2 due to computational cost)
    "N_used_d3_list": [30, 50],

    # Baseline numerical settings for D3
    "D3_base_psi_grid_n": 401,
    "D3_base_barpsi_bisect_iters": 90,
    "D3_base_psi0_buffer": 0.03,

    # Optional numerical sensitivity for D3
    "RUN_D3_NUMERICAL_SENSITIVITY": False,
    "D3_sens_psi_grid_n_list": [201, 401],
    "D3_sens_barpsi_bisect_iters_list": [60, 90],

    # Dynamics
    "T": 60.0,
    "dt": 0.05,
    "reform_times": np.linspace(0.0, 50.0, 11),

    # D1 initial condition mode
    # False: deterministic ψ0 = max(psi_L+0.02, barψ+buffer) (normalized start)
    # True:  sample ψ0 ~ Uniform(barψ+buffer, 0.95) for dispersion across draws
    "D1_SAMPLE_PSI0": False,

    # D2 stimulus sizes (exact Δg*)
    "dGamma_list": [0.005, 0.01, 0.02],

    # D3 escape grid sweeps
    "reform_sizes": np.linspace(0.01, 0.10, 10),
    "start_times": np.linspace(0.0, 50.0, 11),
    "reform_freq_list": [10.0, 20.0, 40.0],
    "max_reforms_list": [5, 10],

    # Institutional drift parameters
    "sigma_range": (0.03, 0.12),
    "omega_range": (0.01, 0.10),

    # Micro return calibration
    "r0_range": (0.00, 0.03),
    "r_at_rhobar_range": (0.02, 0.12),

    # Primitive ranges
    "rho_bar_range": (1.0, 5.0),
    "psi_L_range": (0.0, 0.4),
    "Gamma_range": (0.01, 0.08),
    "kappa_c_range": (0.2, 2.0),
    "c_lambda_range": (0.1, 1.5),
    "eta_lambda_range": (1.0, 3.0),
    "c_Delta_range": (0.1, 1.0),
    "eta_Delta_range": (1.0, 3.0),
    "c_delta_range": (0.05, 0.5),
    "eta_delta_range": (1.0, 3.0),
}

# ----------------------------
# Helpers: saving
# ----------------------------
def save_df(df: pd.DataFrame, tag: str) -> str:
    path = os.path.join(OUT_DIR, f"{MC_NAME}_{tag}_{RUN_TS}.csv")
    df.to_csv(path, index=False)
    return path

def save_fig(fig, name: str) -> str:
    png = os.path.join(FIGS_DIR, f"{name}.png")
    pdf = os.path.join(FIGS_DIR, f"{name}.pdf")
    fig.savefig(png, bbox_inches="tight", dpi=300)
    fig.savefig(pdf, bbox_inches="tight")
    plt.close(fig)
    return pdf


# =============================================================================
# Micro primitives + solvers
# =============================================================================
def r_func(rho, r0, a, b):
    return r0 + a * rho - b * rho**2

def lam_func(rho, c_lam, eta_lam):
    return 1.0 - np.exp(-c_lam * rho**eta_lam)

def lam_prime(rho, c_lam, eta_lam):
    rho = np.asarray(rho, dtype=float)
    out = np.full_like(rho, np.nan, dtype=float)
    valid = np.isfinite(rho) & (rho >= 0.0)
    if np.any(valid):
        f = np.exp(-c_lam * rho[valid]**eta_lam)
        g = c_lam * eta_lam * rho[valid]**(eta_lam - 1.0)
        out[valid] = f * g
    return out

def Delta_func(rho, c_Delta, eta_Delta):
    return c_Delta * rho**eta_Delta

def Delta_prime(rho, c_Delta, eta_Delta):
    rho = np.asarray(rho, dtype=float)
    out = np.full_like(rho, np.nan, dtype=float)
    valid = np.isfinite(rho) & (rho >= 0.0)
    if np.any(valid):
        out[valid] = c_Delta * eta_Delta * rho[valid]**(eta_Delta - 1.0)
    return out

def delta_func(rho, c_delta, eta_delta):
    return c_delta * rho**eta_delta

def B_func(rho, c_lam, eta_lam, c_Delta, eta_Delta):
    lam_p = lam_prime(rho, c_lam, eta_lam)
    lam = lam_func(rho, c_lam, eta_lam)
    D = Delta_func(rho, c_Delta, eta_Delta)
    D_p = Delta_prime(rho, c_Delta, eta_Delta)
    return lam_p * D + lam * D_p

def solve_rho_star(psi, rho_bar, a, b, c_lam, eta_lam, c_Delta, eta_Delta,
                   tol=1e-10, max_iter=250):
    """
    Solve FOC: a - 2b rho - (1-psi) B(rho) = 0 on [0, rho_bar] via bisection.
    Returns (rho*, status).
    """
    def F(rho):
        return a - 2.0 * b * rho - (1.0 - psi) * B_func(rho, c_lam, eta_lam, c_Delta, eta_Delta)

    lo, hi = 0.0, float(rho_bar)
    f_lo = float(F(lo))
    f_hi = float(F(hi))

    if not np.isfinite(f_hi):
        return np.nan, "nonfinite"

    if np.isfinite(f_lo) and abs(f_lo) < tol:
        return 0.0, "corner_low"
    if np.isfinite(f_hi) and abs(f_hi) < tol:
        return hi, "corner_high"

    if np.isfinite(f_lo) and (f_lo * f_hi < 0.0):
        a_lo, a_hi = lo, hi
        for _ in range(max_iter):
            mid = 0.5 * (a_lo + a_hi)
            f_mid = float(F(mid))
            if not np.isfinite(f_mid):
                return np.nan, "nonfinite"
            if abs(f_mid) < tol or (a_hi - a_lo) < tol:
                return mid, "interior"
            if np.sign(f_mid) == np.sign(f_lo):
                a_lo, f_lo = mid, f_mid
            else:
                a_hi = mid
        return mid, "no_converge"

    # no sign change → corners
    if np.isfinite(f_lo) and (f_lo > 0.0) and (f_hi > 0.0):
        return hi, "corner_high"
    if np.isfinite(f_lo) and (f_lo < 0.0) and (f_hi < 0.0):
        return 0.0, "corner_low"

    return np.nan, "no_bracket"

def g_star_stationary(psi, Gamma_eff, kappa_c):
    """
    Positive root of: kappa_c * psi * g^2 + g - Gamma_eff = 0
    """
    if not (np.isfinite(psi) and np.isfinite(Gamma_eff) and np.isfinite(kappa_c)):
        return np.nan
    if Gamma_eff <= 0.0:
        return 0.0
    if psi <= 0.0:
        return float(Gamma_eff)
    disc = 1.0 + 4.0 * kappa_c * psi * Gamma_eff
    if disc <= 0.0:
        return 0.0
    return float((-1.0 + math.sqrt(disc)) / (2.0 * kappa_c * psi))


def pick_psi0(psi_L, barpsi, psi0_buffer, psi_hi, eps_lo=EPS_PSI_L, eps_hi=EPS_PSI_HI):
    """
    Returns a feasible high-basin start ψ0 within [psi_L, psi_hi).
    If barpsi + buffer would exceed the cap, ψ0 is pinned at the cap.
    """
    psi_min = max(float(psi_L) + float(eps_lo), float(barpsi) + float(psi0_buffer))
    psi_cap = float(psi_hi) - float(eps_hi)
    return float(min(psi_min, psi_cap))


# =============================================================================
# Profiles: compute rK(ψ), D(ψ), g*(ψ), W*(ψ) on grid
# =============================================================================
def build_profiles(theta, psi_grid, include_structural_drag: bool):
    """
    Arrays on psi_grid: rho_star, rK, drag, Gamma_eff, gstar, W

    include_structural_drag:
      True  -> Gamma_eff(psi) = Gamma - drag(psi)
      False -> Gamma_eff(psi) = Gamma (Section 4.2 closure only)
    """
    rho0, _ = solve_rho_star(
        0.0, theta["rho_bar"], theta["a"], theta["b"],
        theta["c_lambda"], theta["eta_lambda"],
        theta["c_Delta"], theta["eta_Delta"]
    )
    base_drag = lam_func(rho0, theta["c_lambda"], theta["eta_lambda"]) * delta_func(
        rho0, theta["c_delta"], theta["eta_delta"]
    )

    n = len(psi_grid)
    rho_star = np.full(n, np.nan)
    rK = np.full(n, np.nan)
    drag = np.full(n, np.nan)
    Gamma_eff = np.full(n, np.nan)
    gstar = np.full(n, np.nan)
    W = np.full(n, np.nan)

    for i, psi in enumerate(psi_grid):
        rho, _ = solve_rho_star(
            float(psi), theta["rho_bar"], theta["a"], theta["b"],
            theta["c_lambda"], theta["eta_lambda"],
            theta["c_Delta"], theta["eta_Delta"]
        )
        if not np.isfinite(rho):
            continue

        lam = lam_func(rho, theta["c_lambda"], theta["eta_lambda"])
        Dlt = Delta_func(rho, theta["c_Delta"], theta["eta_Delta"])
        dlt = delta_func(rho, theta["c_delta"], theta["eta_delta"])

        drag_i = lam * dlt - base_drag
        Gamma_eff_i = theta["Gamma"] - (drag_i if include_structural_drag else 0.0)

        g_i = g_star_stationary(float(psi), Gamma_eff_i, theta["kappa_c"])
        rK_i = r_func(rho, theta["r0"], theta["a"], theta["b"]) - (1.0 - psi) * lam * Dlt

        rho_star[i] = rho
        drag[i] = drag_i
        Gamma_eff[i] = Gamma_eff_i
        gstar[i] = g_i
        rK[i] = rK_i
        W[i] = rK_i - g_i

    return {"rho_star": rho_star, "rK": rK, "drag": drag, "Gamma_eff": Gamma_eff, "gstar": gstar, "W": W}

def build_interpolators(psi_grid, prof):
    """
    Build scipy interpolators from profile arrays for efficient repeated evaluation.

    NaN values are filtered before building interpolators, as solve_rho_star()
    may fail at some ψ grid points, leaving NaN entries that would otherwise
    propagate through all subsequent trajectory evaluations.

    Returns dict of interpolator functions callable as: W_interp(psi_value)
    """
    interpolators = {}
    for key in ["rho_star", "rK", "drag", "Gamma_eff", "gstar", "W"]:
        if key not in prof:
            continue
        y = np.asarray(prof[key], dtype=float)
        x = np.asarray(psi_grid, dtype=float)

        # Filter to finite support only
        m = np.isfinite(x) & np.isfinite(y)
        if m.sum() < 2:
            interpolators[key] = lambda z, _nan=np.nan: _nan
            continue

        interpolators[key] = interp1d(
            x[m], y[m],
            kind='linear',
            bounds_error=False,
            fill_value='extrapolate',
            assume_sorted=False
        )
    return interpolators

def find_barpsi(psi_grid, W_grid, psi_L, bisect_iters: int, margin: float = BISTABLE_MARGIN):
    """
    Regime classification and barψ location if bistable.

    Returns: (barpsi, regime, W_L, W_1)

    Regimes:
      BISTABLE:             W*(psi_L) <= -margin < 0 < margin <= W*(1), barψ in (psi_L, 1)
      STRUCTURAL_HIGH_TRAP: W*(psi_L) >= -margin
      STRUCTURAL_LOW:       W*(1) <= margin
      NONFINITE:            W* not computable
      NONMONOTONE:          W*(psi) violates monotonicity

    The margin condition prevents knife-edge bistability under fine grid evaluation.
    """
    W_L = float(np.interp(psi_L, psi_grid, W_grid))
    W_1 = float(np.interp(1.0, psi_grid, W_grid))

    if not (np.isfinite(W_L) and np.isfinite(W_1)):
        return np.nan, "NONFINITE", W_L, W_1

    # Monotonicity check: W*(ψ) should be weakly increasing
    valid = np.isfinite(W_grid)
    if np.sum(valid) > 1:
        dW = np.diff(W_grid[valid])
        min_dW = float(np.min(dW))
        if min_dW < -0.01:
            return np.nan, "NONMONOTONE", W_L, W_1

    if W_L >= -margin:
        return float(psi_L), "STRUCTURAL_HIGH_TRAP", W_L, W_1
    if W_1 <= margin:
        return np.nan, "STRUCTURAL_LOW", W_L, W_1

    # W_L <= -margin < 0 < margin <= W_1: BISTABLE — locate barψ by bisection
    lo, hi = float(psi_L), 1.0
    for _ in range(int(bisect_iters)):
        mid = 0.5 * (lo + hi)
        W_mid = float(np.interp(mid, psi_grid, W_grid))
        if W_mid >= 0.0:
            hi = mid
        else:
            lo = mid
    return float(hi), "BISTABLE", W_L, W_1


# =============================================================================
# Institutional dynamics: 1D quasi-static (Assumption 1)
# =============================================================================
def simulate_psi_path(psi0, psi_L, W_interp, sigma, omega, T, dt, reforms=None):
    """
    Euler integration of:
      dψ/dt = σ * tanh(W*(ψ)/ω),  projected to [psi_L, 1]
    reforms: list of (t_reform, delta_psi) applying ψ <- max(psi_L, ψ - delta_psi)
    """
    n_steps = int(np.ceil(T / dt))
    t = np.linspace(0.0, n_steps * dt, n_steps + 1)
    psi = np.empty(n_steps + 1, dtype=float)
    psi[0] = float(np.clip(psi0, psi_L, 1.0))

    reforms = reforms or []
    reforms_sorted = sorted(reforms, key=lambda x: x[0])
    j = 0

    for k in range(n_steps):
        tk = t[k]

        while j < len(reforms_sorted) and abs(reforms_sorted[j][0] - tk) < 0.5 * dt:
            _, dpsi = reforms_sorted[j]
            psi[k] = max(psi_L, psi[k] - float(dpsi))
            j += 1

        Wk = float(W_interp(psi[k]))
        drift = float(sigma) * math.tanh(Wk / float(omega))
        psi[k + 1] = float(np.clip(psi[k] + dt * drift, psi_L, 1.0))

    return t, psi


# =============================================================================
# Sampling: build a pool of candidate draws
# =============================================================================
def sample_theta():
    rho_bar = rng.uniform(*CFG["rho_bar_range"])
    psi_L = rng.uniform(*CFG["psi_L_range"])
    Gamma = rng.uniform(*CFG["Gamma_range"])
    kappa_c = rng.uniform(*CFG["kappa_c_range"])

    r0 = rng.uniform(*CFG["r0_range"])
    r_at_rhobar = rng.uniform(*CFG["r_at_rhobar_range"])
    a = max(1e-8, 2.0 * (r_at_rhobar - r0) / rho_bar)
    b = a / (2.0 * rho_bar)

    theta = {
        "psi_L": float(psi_L),
        "Gamma": float(Gamma),
        "kappa_c": float(kappa_c),
        "rho_bar": float(rho_bar),
        "r0": float(r0),
        "a": float(a),
        "b": float(b),
        "c_lambda": float(rng.uniform(*CFG["c_lambda_range"])),
        "eta_lambda": float(rng.uniform(*CFG["eta_lambda_range"])),
        "c_Delta": float(rng.uniform(*CFG["c_Delta_range"])),
        "eta_Delta": float(rng.uniform(*CFG["eta_Delta_range"])),
        "c_delta": float(rng.uniform(*CFG["c_delta_range"])),
        "eta_delta": float(rng.uniform(*CFG["eta_delta_range"])),
    }

    sigma = float(rng.uniform(*CFG["sigma_range"]))
    omega = float(rng.uniform(*CFG["omega_range"]))
    return theta, sigma, omega


# =============================================================================
# MAIN
# =============================================================================
def main():
    print("=" * 100)
    print("mc_block_3 — Monte Carlo: Hysteresis, Reform, and Stimulus Trap")
    print(f"Run timestamp: {RUN_TS}")
    print("=" * 100)

    # 1) Build bistable pool under both structural-drag specifications
    pool_rows = []
    regime_counts = {mode: {"BISTABLE": 0, "STRUCTURAL_HIGH_TRAP": 0, "STRUCTURAL_LOW": 0,
                            "NONFINITE": 0, "NONMONOTONE": 0}
                     for mode in CFG["include_structural_drag_list"]}

    tries = 0
    while tries < CFG["MAX_TRIES"] and len(pool_rows) < CFG["N_BISTABLE_POOL_TARGET"]:
        tries += 1
        theta, sigma, omega = sample_theta()

        psi_grid_n = max(CFG["psi_grid_n_list"])
        psi_grid = np.linspace(theta["psi_L"], 1.0, int(psi_grid_n))

        bis_it_ref = int(max(CFG["barpsi_bisect_iters_list"]))

        bistable_any = False
        row = {"theta": theta, "sigma": sigma, "omega": omega}

        for mode in CFG["include_structural_drag_list"]:
            prof = build_profiles(theta, psi_grid, include_structural_drag=bool(mode))
            barpsi, regime, W_L, W_1 = find_barpsi(psi_grid, prof["W"], theta["psi_L"], bisect_iters=bis_it_ref)
            regime_counts[mode][regime] += 1

            row[f"regime_drag_{int(mode)}"] = regime
            row[f"barpsi_drag_{int(mode)}"] = float(barpsi) if np.isfinite(barpsi) else np.nan
            row[f"W_L_drag_{int(mode)}"] = float(W_L) if np.isfinite(W_L) else np.nan
            row[f"W_1_drag_{int(mode)}"] = float(W_1) if np.isfinite(W_1) else np.nan

            if regime == "BISTABLE":
                bistable_any = True

        if bistable_any:
            pool_rows.append(row)

    # Flatten pool to CSV
    flat = []
    for i, r in enumerate(pool_rows):
        th = r["theta"]
        out = {"draw_id": i, "sigma": r["sigma"], "omega": r["omega"], **th}
        for mode in CFG["include_structural_drag_list"]:
            out[f"regime_drag_{int(mode)}"] = r[f"regime_drag_{int(mode)}"]
            out[f"barpsi_drag_{int(mode)}"] = r[f"barpsi_drag_{int(mode)}"]
            out[f"W_L_drag_{int(mode)}"] = r[f"W_L_drag_{int(mode)}"]
            out[f"W_1_drag_{int(mode)}"] = r[f"W_1_drag_{int(mode)}"]
        flat.append(out)

    df_pool = pd.DataFrame(flat)
    pool_path = save_df(df_pool, "bistable_pool")
    print(f"\nBuilt pool: {len(df_pool)} draws (tries={tries})")
    print(f"Saved pool: {pool_path}")

    # 2) Regime summary
    reg_summ = []
    for mode in CFG["include_structural_drag_list"]:
        cnt = regime_counts[mode]
        tot = sum(cnt.values())
        reg_summ.append({
            "include_structural_drag": bool(mode),
            "tries": tries,
            **{f"count_{k}": v for k, v in cnt.items()},
            **{f"share_{k}": (v / tot if tot > 0 else np.nan) for k, v in cnt.items()},
            "note": "D1 and D3 are meaningful only on BISTABLE cases (barψ in (psi_L, 1)). NONMONOTONE: W*(ψ) non-monotonic, flagged and dropped.",
        })
    df_reg = pd.DataFrame(reg_summ)
    save_df(df_reg, "regime_summary")
    print("\nRegime summary:")
    print(df_reg.to_string(index=False))

    if len(df_pool) == 0:
        print("\nNo usable pool. Stopping.")
        return

    # Profile cache keyed by (draw_id, psi_grid_n, include_drag)
    _PROF_CACHE = {}

    def get_profiles(draw_row, psi_grid_n, include_drag):
        """
        Returns (theta, psi_grid, prof, interp).
        Results are cached to avoid redundant recomputation across sweep cells.
        """
        key = (int(draw_row["draw_id"]), int(psi_grid_n), bool(include_drag))
        if key in _PROF_CACHE:
            return _PROF_CACHE[key]
        theta = {k: float(draw_row[k]) for k in [
            "psi_L","Gamma","kappa_c","rho_bar","r0","a","b",
            "c_lambda","eta_lambda","c_Delta","eta_Delta","c_delta","eta_delta"
        ]}
        psi_grid = np.linspace(theta["psi_L"], 1.0, int(psi_grid_n))
        prof = build_profiles(theta, psi_grid, include_structural_drag=bool(include_drag))
        interp = build_interpolators(psi_grid, prof)
        _PROF_CACHE[key] = (theta, psi_grid, prof, interp)
        return _PROF_CACHE[key]

    # =============================================================================
    # D1 sweep: Δψ_crit(t) along no-intervention drift path
    #
    # Initial condition mode (controlled by CFG["D1_SAMPLE_PSI0"]):
    #   False: deterministic ψ0 = max(psi_L+0.02, barψ+buffer) per draw
    #          → Δψ_crit(t) conditional on a normalized high-basin start
    #   True:  sample ψ0 ~ Uniform(barψ+buffer, 0.95) per draw (fixed seed)
    #          → Δψ_crit(t) reflects dispersion in initial conditions
    # =============================================================================
    rows_d1 = []
    T, dt = float(CFG["T"]), float(CFG["dt"])
    reform_times = np.array(CFG["reform_times"], dtype=float)
    sample_psi0_mode = CFG.get("D1_SAMPLE_PSI0", False)

    for psi_grid_n in CFG["psi_grid_n_list"]:
        for bis_it in CFG["barpsi_bisect_iters_list"]:
            for include_drag in CFG["include_structural_drag_list"]:
                regime_col = f"regime_drag_{int(include_drag)}"
                sub = df_pool[df_pool[regime_col] == "BISTABLE"].copy()
                if sub.empty:
                    continue

                for psi0_buffer in CFG["psi0_buffer_list"]:
                    for N_used in CFG["N_used_list"]:
                        subN = sub.head(int(N_used))
                        deltas_by_t = {float(t): [] for t in reform_times}
                        pinned_count = 0
                        total_count = 0

                        for _, r in subN.iterrows():
                            theta, psi_grid, prof, interp = get_profiles(r, psi_grid_n, include_drag)
                            barpsi, regime, _, _ = find_barpsi(psi_grid, prof["W"], theta["psi_L"], bisect_iters=int(bis_it))
                            if regime != "BISTABLE" or not np.isfinite(barpsi):
                                continue

                            psi0_cap = PSI_HI - EPS_PSI_HI

                            if sample_psi0_mode:
                                psi_lo = max(theta["psi_L"] + EPS_PSI_L, float(barpsi) + float(psi0_buffer))
                                psi_hi = psi0_cap
                                if psi_lo >= psi_hi:
                                    psi0 = psi_hi
                                else:
                                    draw_rng = np.random.default_rng(SEED + int(r["draw_id"]))
                                    psi0 = float(draw_rng.uniform(psi_lo, psi_hi))
                            else:
                                psi0 = pick_psi0(theta["psi_L"], barpsi, psi0_buffer, PSI_HI)

                            if (float(barpsi) + float(psi0_buffer)) > psi0_cap:
                                pinned_count += 1
                            total_count += 1

                            W_interp = interp["W"]
                            _, psi_path = simulate_psi_path(
                                psi0, theta["psi_L"], W_interp,
                                sigma=float(r["sigma"]), omega=float(r["omega"]),
                                T=T, dt=dt, reforms=None
                            )

                            for tt in reform_times:
                                k = int(round(tt / dt))
                                k = max(0, min(k, len(psi_path) - 1))
                                deltas_by_t[float(tt)].append(max(0.0, float(psi_path[k] - barpsi)))

                        for tt in reform_times:
                            arr = np.asarray(deltas_by_t[float(tt)], dtype=float)
                            arr = arr[np.isfinite(arr)]
                            if arr.size == 0:
                                continue
                            rows_d1.append({
                                "psi_grid_n": int(psi_grid_n),
                                "barpsi_bisect_iters": int(bis_it),
                                "include_structural_drag": bool(include_drag),
                                "psi0_buffer": float(psi0_buffer),
                                "N_used": int(N_used),
                                "psi0_mode": "sampled" if sample_psi0_mode else "deterministic",
                                "t": float(tt),
                                "median": float(np.median(arr)),
                                "p25": float(np.percentile(arr, 25)),
                                "p75": float(np.percentile(arr, 75)),
                                "N": int(arr.size),
                                "psi0_cap": PSI_HI - EPS_PSI_HI,
                                "N_pinned": pinned_count,
                                "share_pinned": float(pinned_count / total_count) if total_count > 0 else 0.0,
                            })

    df_d1 = pd.DataFrame(rows_d1)
    save_df(df_d1, "deliverable_1_delta_psi_crit_sweep")

    # =============================================================================
    # D2 sweep: exact stimulus attenuation Δg*(ψ; ΔΓ)
    #   For each draw, evaluate the growth response at:
    #     ψ_mod  = max(ψ_L+0.02, barψ + psi0_buffer)  (near the high basin boundary)
    #     ψ_high = 0.95                                 (deep in the high-asymmetry basin)
    #   and compare distributions across draws.
    # =============================================================================
    rows_d2 = []

    for psi_grid_n in CFG["psi_grid_n_list"]:
        for bis_it in CFG["barpsi_bisect_iters_list"]:
            for include_drag in CFG["include_structural_drag_list"]:
                regime_col = f"regime_drag_{int(include_drag)}"
                sub = df_pool[df_pool[regime_col].isin(["BISTABLE", "STRUCTURAL_HIGH_TRAP"])].copy()
                if sub.empty:
                    continue

                for psi0_buffer in CFG["psi0_buffer_list"]:
                    for dGamma in CFG["dGamma_list"]:
                        for N_used in CFG["N_used_list"]:
                            subN = sub.head(int(N_used))

                            dg_mod_list, dg_high_list = [], []
                            g_mod_list, g_high_list = [], []
                            pinned_count = 0
                            total_count = 0

                            for _, r in subN.iterrows():
                                theta, psi_grid, prof, interp = get_profiles(r, psi_grid_n, include_drag)
                                barpsi, regime, _, _ = find_barpsi(
                                    psi_grid, prof["W"], theta["psi_L"], bisect_iters=int(bis_it)
                                )
                                if regime not in ["BISTABLE", "STRUCTURAL_HIGH_TRAP"]:
                                    continue
                                if not np.isfinite(barpsi):
                                    continue

                                psi_mod = pick_psi0(theta["psi_L"], barpsi, psi0_buffer, PSI_HI)
                                psi_high = float(PSI_HI) - float(EPS_PSI_HI)

                                psi0_cap = psi_high
                                if (float(barpsi) + float(psi0_buffer)) > psi0_cap:
                                    pinned_count += 1
                                total_count += 1

                                g_mod = float(interp["gstar"](psi_mod))
                                Ge_mod = float(interp["Gamma_eff"](psi_mod))
                                g_high = float(interp["gstar"](psi_high))
                                Ge_high = float(interp["Gamma_eff"](psi_high))

                                # Exact Δg*
                                g_mod_1 = g_star_stationary(psi_mod, Ge_mod + float(dGamma), theta["kappa_c"])
                                g_high_1 = g_star_stationary(psi_high, Ge_high + float(dGamma), theta["kappa_c"])
                                dg_mod = float(g_mod_1 - g_mod)
                                dg_high = float(g_high_1 - g_high)

                                if np.isfinite(dg_mod) and np.isfinite(dg_high):
                                    dg_mod_list.append(dg_mod)
                                    dg_high_list.append(dg_high)
                                    g_mod_list.append(g_mod)
                                    g_high_list.append(g_high)

                            if len(dg_mod_list) == 0:
                                continue

                            dg_mod_arr = np.asarray(dg_mod_list)
                            dg_high_arr = np.asarray(dg_high_list)

                            rows_d2.append({
                                "psi_grid_n": int(psi_grid_n),
                                "barpsi_bisect_iters": int(bis_it),
                                "include_structural_drag": bool(include_drag),
                                "psi0_buffer": float(psi0_buffer),
                                "dGamma": float(dGamma),
                                "N_used": int(N_used),

                                "dg_mod_median": float(np.median(dg_mod_arr)),
                                "dg_mod_p25": float(np.percentile(dg_mod_arr, 25)),
                                "dg_mod_p75": float(np.percentile(dg_mod_arr, 75)),

                                "dg_high_median": float(np.median(dg_high_arr)),
                                "dg_high_p25": float(np.percentile(dg_high_arr, 25)),
                                "dg_high_p75": float(np.percentile(dg_high_arr, 75)),

                                "attenuation_ratio_median": float(np.median(dg_high_arr / np.maximum(dg_mod_arr, 1e-12))),
                                "N_eval": int(len(dg_mod_arr)),

                                "psi0_cap": PSI_HI - EPS_PSI_HI,
                                "N_pinned": pinned_count,
                                "share_pinned": float(pinned_count / total_count) if total_count > 0 else 0.0,
                            })

    df_d2 = pd.DataFrame(rows_d2)
    save_df(df_d2, "deliverable_2_stimulus_attenuation_sweep")

    # =============================================================================
    # D3 sweep: escape probability grid
    # Policy sweep axes: include_drag, reform_freq, max_reforms, reform_size, t_start
    # Numerical knobs held at conservative baseline values declared in CFG.
    # =============================================================================
    rows_d3 = []
    reform_sizes = np.array(CFG["reform_sizes"], dtype=float)
    start_times = np.array(CFG["start_times"], dtype=float)

    def run_d3_block(psi_grid_n, bis_it, psi0_buffer, include_drag, N_used_d3):
        regime_col = f"regime_drag_{int(include_drag)}"
        sub = df_pool[df_pool[regime_col] == "BISTABLE"].copy()
        if sub.empty:
            return

        subN = sub.head(int(N_used_d3))

        for reform_freq in CFG["reform_freq_list"]:
            for max_reforms in CFG["max_reforms_list"]:
                for dpsi in reform_sizes:
                    for t0 in start_times:
                        esc = 0
                        tot = 0
                        n_ref_used_list = []
                        pinned_count = 0

                        for _, r in subN.iterrows():
                            theta, psi_grid, prof, interp = get_profiles(r, psi_grid_n, include_drag)
                            barpsi, regime, _, _ = find_barpsi(
                                psi_grid, prof["W"], theta["psi_L"], bisect_iters=int(bis_it)
                            )
                            if regime != "BISTABLE" or not np.isfinite(barpsi):
                                continue

                            psi0 = pick_psi0(theta["psi_L"], barpsi, psi0_buffer, PSI_HI)

                            psi0_cap = PSI_HI - EPS_PSI_HI
                            if (float(barpsi) + float(psi0_buffer)) > psi0_cap:
                                pinned_count += 1

                            n_ref = int(max(0.0, (T - float(t0)) / float(reform_freq)))
                            n_ref = min(int(max_reforms), n_ref)
                            reforms = [(float(t0 + k * reform_freq), float(dpsi)) for k in range(n_ref)]
                            n_ref_used_list.append(n_ref)

                            W_interp = interp["W"]

                            if n_ref == 0:
                                escaped = False
                            else:
                                # Simulate to last reform time; escape is determined at that point
                                t_last = reforms[-1][0]
                                T_sim = min(T, t_last + dt)
                                _, psi_path = simulate_psi_path(
                                    psi0, theta["psi_L"], W_interp,
                                    sigma=float(r["sigma"]), omega=float(r["omega"]),
                                    T=T_sim, dt=dt, reforms=reforms
                                )

                                k_last = int(round(t_last / dt))
                                k_last = max(0, min(k_last, len(psi_path) - 1))
                                escaped = bool(psi_path[k_last] < barpsi)

                            esc += int(escaped)
                            tot += 1

                        P = (esc / tot) if tot > 0 else np.nan
                        rows_d3.append({
                            "psi_grid_n": int(psi_grid_n),
                            "barpsi_bisect_iters": int(bis_it),
                            "include_structural_drag": bool(include_drag),
                            "psi0_buffer": float(psi0_buffer),
                            "N_used": int(N_used_d3),
                            "reform_size": float(dpsi),
                            "t_start": float(t0),
                            "reform_freq": float(reform_freq),
                            "max_reforms": int(max_reforms),
                            "n_reforms_used_median": float(np.median(n_ref_used_list)) if n_ref_used_list else np.nan,
                            "P_escape": float(P),
                            "N_eval": int(tot),
                            "psi0_cap": PSI_HI - EPS_PSI_HI,
                            "N_pinned": pinned_count,
                            "share_pinned": float(pinned_count / tot) if tot > 0 else 0.0,
                        })

    # Baseline D3 run
    for include_drag in CFG["include_structural_drag_list"]:
        for N_used_d3 in CFG["N_used_d3_list"]:
            run_d3_block(
                psi_grid_n=int(CFG["D3_base_psi_grid_n"]),
                bis_it=int(CFG["D3_base_barpsi_bisect_iters"]),
                psi0_buffer=float(CFG["D3_base_psi0_buffer"]),
                include_drag=bool(include_drag),
                N_used_d3=int(N_used_d3),
            )

    # Optional numerical sensitivity sweep for D3
    if CFG.get("RUN_D3_NUMERICAL_SENSITIVITY", False):
        for include_drag in CFG["include_structural_drag_list"]:
            for N_used_d3 in CFG["N_used_d3_list"]:
                for psi_grid_n in CFG["D3_sens_psi_grid_n_list"]:
                    for bis_it in CFG["D3_sens_barpsi_bisect_iters_list"]:
                        run_d3_block(
                            psi_grid_n=int(psi_grid_n),
                            bis_it=int(bis_it),
                            psi0_buffer=float(CFG["D3_base_psi0_buffer"]),
                            include_drag=bool(include_drag),
                            N_used_d3=int(N_used_d3),
                        )

    df_d3 = pd.DataFrame(rows_d3)
    save_df(df_d3, "deliverable_3_escape_grid_sweep")

    # =============================================================================
    # FIGURES
    # =============================================================================
    print("\nGenerating figures...")

    # D1: Δψ_crit(t) over time (median + IQR)
    if not df_d1.empty:
        ref_cfg = df_d1[
            (df_d1["psi_grid_n"] == max(CFG["psi_grid_n_list"])) &
            (df_d1["barpsi_bisect_iters"] == max(CFG["barpsi_bisect_iters_list"])) &
            (df_d1["psi0_buffer"] == 0.03) &
            (df_d1["N_used"] == max(CFG["N_used_list"]))
        ]

        if not ref_cfg.empty:
            fig, axes = plt.subplots(1, 2, figsize=(12, 4))

            for i, drag_mode in enumerate(CFG["include_structural_drag_list"]):
                ax = axes[i]
                subset = ref_cfg[ref_cfg["include_structural_drag"] == drag_mode]

                if not subset.empty:
                    t = subset["t"].values
                    med = subset["median"].values
                    p25 = subset["p25"].values
                    p75 = subset["p75"].values

                    ax.plot(t, med, 'o-', linewidth=2, label='Median')
                    ax.fill_between(t, p25, p75, alpha=0.3, label='IQR')
                    ax.set_xlabel("Time (years)", fontsize=11)
                    ax.set_ylabel("Δψ_crit(t)", fontsize=11)
                    ax.set_title(f"Structural Drag: {drag_mode}", fontsize=12)
                    ax.legend(fontsize=10)
                    ax.grid(True, alpha=0.3)

            plt.tight_layout()
            save_fig(fig, "D1_delta_psi_crit_vs_time")

    # D2: Stimulus attenuation (ψ_mod vs ψ_high)
    if not df_d2.empty:
        ref_cfg = df_d2[
            (df_d2["psi_grid_n"] == max(CFG["psi_grid_n_list"])) &
            (df_d2["barpsi_bisect_iters"] == max(CFG["barpsi_bisect_iters_list"])) &
            (df_d2["psi0_buffer"] == 0.03) &
            (df_d2["N_used"] == max(CFG["N_used_list"]))
        ]

        if not ref_cfg.empty:
            fig, axes = plt.subplots(1, 2, figsize=(12, 4))

            for i, drag_mode in enumerate(CFG["include_structural_drag_list"]):
                ax = axes[i]
                subset = ref_cfg[ref_cfg["include_structural_drag"] == drag_mode]

                if not subset.empty:
                    dGamma_vals = subset["dGamma"].values
                    dg_mod = subset["dg_mod_median"].values
                    dg_high = subset["dg_high_median"].values

                    x = np.arange(len(dGamma_vals))
                    width = 0.35

                    ax.bar(x - width/2, dg_mod, width, label='ψ_mod (near high basin)',
                           alpha=0.8, color='0.7')
                    ax.bar(x + width/2, dg_high, width, label='ψ_high (extreme)',
                           alpha=0.8, color='0.3')
                    ax.set_xlabel("ΔΓ stimulus size", fontsize=11)
                    ax.set_ylabel("Δg* (median)", fontsize=11)
                    ax.set_title(f"Structural Drag: {drag_mode}", fontsize=12)
                    ax.set_xticks(x)
                    ax.set_xticklabels([f"{v:.3f}" for v in dGamma_vals])
                    ax.legend(fontsize=10)
                    ax.grid(True, alpha=0.3, axis='y')

            plt.tight_layout()
            save_fig(fig, "D2_stimulus_attenuation_comparison")

    # D3: Escape probability heatmaps
    if not df_d3.empty:
        baseline = df_d3[
            (df_d3["psi_grid_n"] == CFG["D3_base_psi_grid_n"]) &
            (df_d3["barpsi_bisect_iters"] == CFG["D3_base_barpsi_bisect_iters"]) &
            (df_d3["psi0_buffer"] == CFG["D3_base_psi0_buffer"]) &
            (df_d3["N_used"] == max(CFG["N_used_d3_list"]))
        ]

        if not baseline.empty:
            for drag_mode in CFG["include_structural_drag_list"]:
                for reform_freq in CFG["reform_freq_list"]:
                    for max_ref in CFG["max_reforms_list"]:
                        subset = baseline[
                            (baseline["include_structural_drag"] == drag_mode) &
                            (baseline["reform_freq"] == reform_freq) &
                            (baseline["max_reforms"] == max_ref)
                        ]

                        if subset.empty:
                            continue

                        pivot = subset.pivot_table(
                            values='P_escape',
                            index='reform_size',
                            columns='t_start',
                            aggfunc='first'
                        )

                        if pivot.empty:
                            continue

                        fig, ax = plt.subplots(figsize=(10, 6))
                        im = ax.imshow(pivot.values, aspect='auto', origin='lower',
                                      cmap='gray', vmin=0.0, vmax=1.0)

                        ax.set_xticks(np.arange(len(pivot.columns)))
                        ax.set_yticks(np.arange(len(pivot.index)))
                        ax.set_xticklabels([f"{v:.0f}" for v in pivot.columns], fontsize=9)
                        ax.set_yticklabels([f"{v:.2f}" for v in pivot.index], fontsize=9)

                        ax.set_xlabel("Reform start time (years)", fontsize=11)
                        ax.set_ylabel("Reform size (Δψ)", fontsize=11)
                        ax.set_title(
                            f"Escape Probability | Drag={drag_mode} | Freq={reform_freq:.0f}yr | Cap={max_ref}",
                            fontsize=12
                        )

                        cbar = plt.colorbar(im, ax=ax)
                        cbar.set_label('P(escape)', fontsize=11)

                        plt.tight_layout()
                        fname = f"D3_escape_prob_drag{int(drag_mode)}_freq{int(reform_freq)}_cap{max_ref}"
                        save_fig(fig, fname)

    print(f"Figures saved to: {FIGS_DIR}")

    # Config dump
    def _jsonable(x):
        if isinstance(x, np.ndarray):
            return x.tolist()
        if isinstance(x, (np.floating, np.integer)):
            return x.item()
        return x

    config_dump = {
        "MC_NAME": MC_NAME,
        "RUN_TS": RUN_TS,
        "SEED": SEED,
        "CFG": {k: _jsonable(v) for k, v in CFG.items()},
        "platform": {
            "python": platform.python_version(),
            "platform": platform.platform(),
            "numpy": np.__version__,
            "pandas": pd.__version__,
        },
    }
    with open(os.path.join(OUT_DIR, "config_dump.json"), "w", encoding="utf-8") as f:
        json.dump(config_dump, f, indent=2)

    print("\n" + "=" * 100)
    print("mc_block_3 — COMPLETE")
    print("=" * 100)
    print(f"Outputs in: {OUT_DIR}")


if __name__ == "__main__":
    main()